# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Mohil Ahuja  
**`Roll Number`:** U20230121 
**`GitHub Branch`:** Mohil_U20230121  

# Imports and Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [3]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [18]:
# Remove duplicates
news_df.drop_duplicates(inplace=True)
train_users.drop_duplicates(inplace=True)
test_users.drop_duplicates(inplace=True)

# Handle missing values
news_df.fillna(method="ffill", inplace=True)
train_users.fillna(method="ffill", inplace=True)
test_users.fillna(method="ffill", inplace=True)


C:\Users\mohil\AppData\Local\Temp\ipykernel_8688\2465111671.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  news_df.fillna(method="ffill", inplace=True)
C:\Users\mohil\AppData\Local\Temp\ipykernel_8688\2465111671.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train_users.fillna(method="ffill", inplace=True)
C:\Users\mohil\AppData\Local\Temp\ipykernel_8688\2465111671.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_users.fillna(method="ffill", inplace=True)


In [19]:
print(train_users.columns)


Index(['user_id', 'age', 'income', 'clicks', 'purchase_amount', 'label',
       'user_label'],
      dtype='object')


In [20]:
user_map = {
    "User1": 0,
    "User2": 1,
    "User3": 2
}

train_users["user_label"] = train_users["label"].map(user_map)
test_users["user_label"] = test_users["label"].map(user_map)


In [21]:
print(train_users["user_label"].value_counts())



Series([], Name: count, dtype: int64)


In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_users["user_label"] = le.fit_transform(train_users["label"])
test_users["user_label"] = le.transform(test_users["label"])


In [23]:
for col in train_users.columns:
    if train_users[col].dtype == "object" and col != "label":
        encoder = LabelEncoder()
        train_users[col] = encoder.fit_transform(train_users[col])
        test_users[col] = encoder.transform(test_users[col])


In [24]:
print(train_users["user_label"].unique())
print(train_users["user_label"].isna().sum())


[2 1 0]
0


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


In [26]:
X_train = train_users.drop(["label", "user_label"], axis=1)
y_train = train_users["user_label"]

X_test = test_users.drop(["label", "user_label"], axis=1)
y_test = test_users["user_label"]




In [28]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        multi_class="auto",
        class_weight="balanced"
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("User Classification Accuracy:", accuracy)


User Classification Accuracy: 0.317


C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [30]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Best Parameters:", grid.best_params_)
print("User Classification Accuracy:", accuracy)


Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
User Classification Accuracy: 0.306


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
